# 面试题：长程 Agent Harness 怎样隔离与恢复？

Harness 要把任务放进带 fixture、task namespace、attempt、deadline 与 checkpoint 的环境。恢复时拒绝旧 attempt 的晚到结果，清理任务资源后才运行下一个 fixture。它不同于普通 event replay：重点验证中断、恢复、隔离和晚到副作用。

## 真实案例

两个租户各运行三次代码修复任务，事件包含草稿、超时、重启与晚到结果。

## 基线

基线用一个全局目录和全局任务 ID 接收所有结果。

## 结果解读

手写 Harness 用 task/attempt 匹配与 fixture namespace 隔离。

## 失败案例

旧 attempt 的成功结果不能覆盖重启后的失败状态。

In [1]:
events = [{'id':'L1','task':'t1','tenant':'acme','attempt':1,'kind':'draft','status':'ok'}, {'id':'L2','task':'t1','tenant':'acme','attempt':1,'kind':'timeout','status':'timeout'}, {'id':'L3','task':'t1','tenant':'acme','attempt':2,'kind':'resume','status':'ok'}, {'id':'L4','task':'t1','tenant':'acme','attempt':1,'kind':'late_result','status':'ok'}, {'id':'L5','task':'t2','tenant':'beta','attempt':1,'kind':'draft','status':'ok'}, {'id':'L6','task':'t2','tenant':'beta','attempt':1,'kind':'cleanup','status':'ok'}]  # 构造六条跨租户、跨 attempt 的长程任务事件。
print('Harness 事件:', events)  # 输出任务、租户、尝试号和事件类型。
print('教学说明：fixture 是离线替身资源，真实长程执行还需容器、网络和身份隔离。')  # 说明实验边界。

Harness 事件: [{'id': 'L1', 'task': 't1', 'tenant': 'acme', 'attempt': 1, 'kind': 'draft', 'status': 'ok'}, {'id': 'L2', 'task': 't1', 'tenant': 'acme', 'attempt': 1, 'kind': 'timeout', 'status': 'timeout'}, {'id': 'L3', 'task': 't1', 'tenant': 'acme', 'attempt': 2, 'kind': 'resume', 'status': 'ok'}, {'id': 'L4', 'task': 't1', 'tenant': 'acme', 'attempt': 1, 'kind': 'late_result', 'status': 'ok'}, {'id': 'L5', 'task': 't2', 'tenant': 'beta', 'attempt': 1, 'kind': 'draft', 'status': 'ok'}, {'id': 'L6', 'task': 't2', 'tenant': 'beta', 'attempt': 1, 'kind': 'cleanup', 'status': 'ok'}]
教学说明：fixture 是离线替身资源，真实长程执行还需容器、网络和身份隔离。


In [2]:
global_results = {}  # 初始化没有 task/attempt 约束的错误全局结果表。
for event in events:  # 按到达顺序接收所有事件。
    global_results[event['task']] = event['status']  # 让晚到结果覆盖当前任务状态。
print('全局基线状态:', global_results)  # 输出无法区分重启前后 attempt 的污染结果。
print('基线问题：t1 的 L4 晚到结果会覆盖 attempt=2 的状态。')  # 明确串扰来源。

全局基线状态: {'t1': 'ok', 't2': 'ok'}
基线问题：t1 的 L4 晚到结果会覆盖 attempt=2 的状态。


In [3]:
runtime = {'t1':2,'t2':1}  # 保存每个任务当前允许写入的 attempt。
ledger = {}  # 初始化按租户和任务隔离的事件账本。
def accept(event):  # 定义 Harness 对事件归属和晚到结果的门禁。
    key = event['tenant'] + ':' + event['task']  # 组成隔离 fixture 的命名空间键。
    if event['attempt'] != runtime[event['task']]:  # 检查结果是否来自当前 attempt。
        return 'ignored_stale'  # 拒绝重启前晚到的旧结果。
    ledger.setdefault(key, []).append(event)  # 在隔离命名空间中追加当前事件。
    return 'accepted'  # 返回当前 attempt 的有效结果。

In [4]:
results = [(event['id'], accept(event)) for event in events]  # 对六条长程任务事件运行 fixture/attempt 门禁。
print('id | Harness 结论')  # 输出事件接收结果表标题。
for item in results:  # 遍历 accepted 与 ignored_stale 结论。
    print(item[0], item[1])  # 输出当前事件是否可写入任务账本。
print('隔离账本键:', sorted(ledger))  # 输出 acme 与 beta 互不污染的资源命名空间。
print('t1 当前事件数:', len(ledger['acme:t1']))  # 输出只保留当前 attempt 的任务事件。

id | Harness 结论
L1 ignored_stale
L2 ignored_stale
L3 accepted
L4 ignored_stale
L5 accepted
L6 accepted
隔离账本键: ['acme:t1', 'beta:t2']
t1 当前事件数: 1


In [5]:
wrong = global_results['t1']  # 读取全局基线被晚到结果污染后的最终状态。
fixed = dict(results)['L4']  # 读取 Harness 对旧 attempt 晚到结果的结论。
print('失败案例 L4：全局状态=', wrong, '，Harness=', fixed)  # 展示恢复后拒绝旧 attempt 写入。
print('生产差距：还要实现真实 deadline、取消传播、资源清理、checkpoint 存储、配额与按任务收集 trace。')  # 说明 Harness 的生产责任。

失败案例 L4：全局状态= ok ，Harness= ignored_stale
生产差距：还要实现真实 deadline、取消传播、资源清理、checkpoint 存储、配额与按任务收集 trace。


In [6]:
assert dict(results)['L4'] == 'ignored_stale'  # 验证旧 attempt 的晚到结果不会写入恢复后的任务。
assert 'acme:t1' in ledger and 'beta:t2' in ledger  # 验证两个 fixture 命名空间隔离存在。
assert len(ledger['acme:t1']) == 1  # 验证 t1 仅保留 attempt=2 的当前事件。